# Real-World Dataset Analysis Project
**Data Science Consultant Internship — Task 3**

Dataset: **Breast Cancer Wisconsin (Diagnostic)** — measurements of cell nuclei from digitized images of breast mass fine needle aspirates, used to classify tumors as malignant or benign. This notebook covers dataset selection, exploration, cleaning, feature engineering, visualization, and insight generation.

## 1. Dataset Selection & Exploration
The dataset is sourced from scikit-learn's built-in repository (originally UCI Machine Learning Repository). Business/analytical context: supporting early, data-driven breast cancer diagnosis from measurable cell characteristics instead of relying on manual interpretation alone.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

# Load dataset and select a readable subset of 'mean' measurement columns
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df['diagnosis'] = df['target'].map({0: 'malignant', 1: 'benign'})
keep = ['mean radius','mean texture','mean perimeter','mean area','mean smoothness',
        'mean compactness','mean concavity','mean symmetry','diagnosis']
df = df[keep]
df.columns = ['radius_mean','texture_mean','perimeter_mean','area_mean','smoothness_mean',
              'compactness_mean','concavity_mean','symmetry_mean','diagnosis']
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
# Check structure: missing values, duplicates, class balance
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nDiagnosis distribution:\n", df['diagnosis'].value_counts())

**Note:** this notebook introduces a small number of realistic data-quality issues (missing values, inconsistent category labels, duplicate rows) on top of the otherwise clean benchmark dataset, purely to give the cleaning stage something real to do — exactly as a messy real-world extract would look.

In [ ]:
# Simulate realistic data-quality issues for the cleaning exercise
np.random.seed(7)
for col, frac in [('texture_mean', 0.03), ('smoothness_mean', 0.02), ('symmetry_mean', 0.015)]:
    idx = df.sample(frac=frac, random_state=1).index
    df.loc[idx, col] = np.nan

inconsistent_idx = df.sample(n=6, random_state=2).index
df.loc[inconsistent_idx, 'diagnosis'] = df.loc[inconsistent_idx, 'diagnosis'].map(
    {'benign': 'Benign ', 'malignant': 'MALIGNANT'})

dup_rows = df.sample(n=5, random_state=3)
df = pd.concat([df, dup_rows], ignore_index=True)

print("Shape after introducing issues:", df.shape)
print("Missing values:\n", df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print("Diagnosis labels:", df['diagnosis'].unique())

## 2. Data Cleaning & Preprocessing
Handle missing values, duplicates, inconsistent records, and incorrect/incorrectly-formatted categories.

In [ ]:
# Standardize inconsistent category labels (whitespace, casing)
df['diagnosis'] = df['diagnosis'].str.strip().str.lower()

# Remove exact duplicate rows
before = df.shape[0]
df = df.drop_duplicates()
print(f"Removed {before - df.shape[0]} duplicate rows")

# Impute missing numeric values with the column median
num_cols = [c for c in df.columns if c != 'diagnosis']
for col in num_cols:
    missing = df[col].isnull().sum()
    if missing > 0:
        df[col] = df[col].fillna(df[col].median())
        print(f"Imputed {missing} missing values in '{col}' with median")

print("\nFinal shape:", df.shape)
print("Remaining missing values:", df.isnull().sum().sum())

In [ ]:
# Outlier check via IQR on area_mean
Q1, Q3 = df['area_mean'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outliers = df[(df['area_mean'] < lower) | (df['area_mean'] > upper)]
print(f"Outliers in area_mean (IQR method): {len(outliers)} rows")

## 3. Feature Engineering & Data Manipulation
Create new, more interpretable features and manipulate the data to support analysis.

In [ ]:
# Feature engineering: ratio features and a readable size category
df['area_to_perimeter'] = df['area_mean'] / df['perimeter_mean']
df['compactness_ratio'] = df['compactness_mean'] / df['smoothness_mean']

bins = [0, 12, 15, 18, 100]
labels = ['Small', 'Medium', 'Large', 'Very Large']
df['radius_category'] = pd.cut(df['radius_mean'], bins=bins, labels=labels)

df[['radius_mean','radius_category','area_to_perimeter','compactness_ratio']].head()

In [ ]:
# Manipulation: filter, sort, group
large_malignant = df[(df['radius_category'].isin(['Large','Very Large'])) & (df['diagnosis']=='malignant')]
print("Large/very-large malignant tumors:", len(large_malignant))

df.sort_values('area_mean', ascending=False)[['diagnosis','radius_mean','area_mean']].head()

In [ ]:
# Group means by diagnosis
df.groupby('diagnosis')[num_cols].mean().round(2)

## 4. Exploratory Data Analysis & Visualizations
Bar chart, histogram, line chart, scatter plot, box plot, and correlation heatmap.

In [ ]:
# Bar chart: case counts by diagnosis
counts = df['diagnosis'].value_counts()
plt.figure(figsize=(6,5))
plt.bar(counts.index, counts.values, color=['#55A868','#C44E52'])
plt.title('Number of Cases by Diagnosis')
plt.ylabel('Count')
plt.show()

In [ ]:
# Histogram: radius_mean distribution by diagnosis
plt.figure(figsize=(7,5))
for d, c in {'benign':'#55A868','malignant':'#C44E52'}.items():
    sns.histplot(df[df['diagnosis']==d]['radius_mean'], bins=25, color=c, label=d, alpha=0.6)
plt.title('Distribution of Mean Radius by Diagnosis')
plt.xlabel('Radius Mean')
plt.legend()
plt.show()

In [ ]:
# Line chart: perimeter & area trend across size categories
order = ['Small','Medium','Large','Very Large']
trend = df.groupby('radius_category', observed=True)[['area_mean','perimeter_mean']].mean().reindex(order)
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(trend.index, trend['perimeter_mean'], marker='o', label='Perimeter Mean')
ax2 = ax.twinx()
ax2.plot(trend.index, trend['area_mean'], marker='s', color='orange', label='Area Mean')
ax.set_title('Perimeter & Area Trend Across Tumor Size Categories')
fig.legend(loc='upper left', bbox_to_anchor=(0.12,0.88))
plt.show()

In [ ]:
# Scatter plot: radius vs concavity by diagnosis
plt.figure(figsize=(7,5.5))
for d, c in {'benign':'#55A868','malignant':'#C44E52'}.items():
    sub = df[df['diagnosis']==d]
    plt.scatter(sub['radius_mean'], sub['concavity_mean'], label=d, color=c, alpha=0.6, edgecolor='k')
plt.xlabel('Radius Mean')
plt.ylabel('Concavity Mean')
plt.title('Radius vs Concavity by Diagnosis')
plt.legend()
plt.show()

In [ ]:
# Box plot: area_mean by diagnosis (outlier detection)
plt.figure(figsize=(6,5.5))
sns.boxplot(data=df, x='diagnosis', y='area_mean', palette={'benign':'#55A868','malignant':'#C44E52'})
plt.title('Area Mean Distribution by Diagnosis')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(7,6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Tumor Features')
plt.show()

## 5. Documentation & Insights

In [ ]:
df['diagnosis_binary'] = (df['diagnosis']=='benign').astype(int)
corr_target = df[num_cols + ['diagnosis_binary']].corr()['diagnosis_binary'].drop('diagnosis_binary').sort_values()
print("Correlation of each feature with benign diagnosis:")
print(corr_target.round(3))

### Key Insights

- **Dataset:** 569 patient records, 8 core tumor measurements, cleaned from 574 raw rows (5 duplicates removed, 37 missing values imputed, inconsistent diagnosis labels standardized).
- **Class balance:** 357 benign vs. 212 malignant cases (~63% / 37%) — a moderate imbalance worth accounting for in any predictive modelling.
- **Strongest diagnostic signals:** perimeter_mean (r = -0.74), radius_mean (r = -0.73), and area_mean (r = -0.71) are the most strongly correlated with diagnosis — larger tumors are substantially more likely to be malignant.
- **Clear separation by size:** malignant tumors average a radius of 17.5 vs. 12.2 for benign, and nearly double the area (978 vs. 463) — visible directly in the histogram, scatter plot, and box plot.
- **Shape matters too:** concavity_mean and compactness_mean are also elevated in malignant cases, suggesting irregular, less smooth cell boundaries are a secondary diagnostic signal alongside raw size.
- **Business takeaway:** a simple rule combining radius/perimeter/area with concavity could serve as a fast, interpretable triage signal ahead of (not instead of) full clinical diagnosis.